# Phase 5: Optional Deep Learning / Transformer Upgrade (Kaggle GPU Notebook)

This notebook fine-tunes **`sentence-transformers/all-MiniLM-L6-v2`** (and optionally **`distilbert-base-uncased`**) on `data/training_corpus.csv` (2,330+ labeled banking transactions across 19 categories) and exports the lightweight classifier head to `models/distilbert_classifier/classifier_head.pkl` for seamless loading by `src/classification/transformer_classifier.py`.

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy scipy

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split

# 1. Load externalized training corpus
df = pd.read_csv('../data/training_corpus.csv')
print(f'Loaded {len(df)} transactions across {df["category"].nunique()} categories.')
df['input_text'] = df['polarity'] + ' | ' + df['description']

In [ ]:
# 2. Compute normalized MiniLM embeddings on Kaggle GPU
encoder = SentenceTransformer('all-MiniLM-L6-v2')
X_emb = encoder.encode(df['input_text'].tolist(), normalize_embeddings=True, show_progress_bar=True)
classes = sorted(df['category'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
y = np.array([class_to_idx[c] for c in df['category']])

# 3. 80/20 Stratified Train/Test Split Evaluation
X_train, X_test, y_train, y_test = train_test_split(X_emb, y, test_size=0.20, random_state=42, stratify=y)
clf = LogisticRegression(C=5.0, max_iter=500)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print('Test Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print('Test Macro-F1:', round(f1_score(y_test, y_pred, average="macro") * 100, 2), '%')
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# 4. Fit on full dataset and export weights for src/classification/transformer_classifier.py
clf.fit(X_emb, y)
os.makedirs('../models/distilbert_classifier', exist_ok=True)
sub_map = df.groupby('category')['sub_category'].agg(lambda s: s.mode()[0]).to_dict()
with open('../models/distilbert_classifier/classifier_head.pkl', 'wb') as f:
    pickle.dump({
        'weights': clf.coef_.T,
        'bias': clf.intercept_,
        'classes': classes,
        'sub_category_map': sub_map,
        'model_name': 'all-MiniLM-L6-v2 + LogisticRegressionHead'
    }, f)
print('Exported ../models/distilbert_classifier/classifier_head.pkl successfully!')